# Run FedRBE on Quartet proteomics multi-batch dataset

This notebook runs the federated batch effect correction (FedRBE) on the proteomics multi-batch dataset
(Quartet, PXD045065) using the FeatureCloud testing environment.

**Prerequisites:**
- Docker running
- `featurecloud.ai/bcorrect:latest` image available (pull with `docker pull featurecloud.ai/bcorrect:latest`)
- Data prepared by `01_data_prep_RBE.ipynb` (center folders in `before/`)
- Python packages: `FeatureCloud`, `pandas`, `pyyaml` (from `requirements.txt`)

**What this does:**
1. Defines the experiment for 4 centers (APT, FDU, NVG, BGI) with per-center config
2. Runs FedRBE via the FeatureCloud controller (non-SMPC and SMPC variants)
3. Extracts and concatenates per-client results into `after/FedApp_corrected_data.tsv` and `after/FedApp_corrected_data_smpc.tsv`

This mirrors the logic in `generate_fedrbe_corrected_datasets.py` but is self-contained for the multi-batch dataset.

In [19]:
import os
import json
import zipfile
import time
import pandas as pd
from copy import deepcopy

import sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
from evaluation_utils import featurecloud_api_extension as util

## Settings

In [20]:
# Paths
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
data_dir = os.path.join(repo_root, 'evaluation_data')
multibatch_dir = os.path.join(data_dir, 'proteomics_multibatch')
before_dir = os.path.join(multibatch_dir, 'before')
after_dir = os.path.join(multibatch_dir, 'after')
os.makedirs(after_dir, exist_ok=True)

app_image_name = 'featurecloud.ai/bcorrect:latest'

# Centers and their properties
# APT and FDU have 2 internal batches (DDA + DIA) -> batch_col: batch
# NVG and BGI have 1 batch each -> batch_col: null
centers = ['APT', 'FDU', 'NVG', 'BGI']
multi_batch_centers = {'APT', 'FDU'}  # centers with >1 internal batch

print(f'Data dir: {data_dir}')
print(f'Before dir: {before_dir}')
print(f'After dir: {after_dir}')
print(f'Centers: {centers}')

Data dir: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data
Before dir: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/before
After dir: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/after
Centers: ['APT', 'FDU', 'NVG', 'BGI']


## Define experiment configuration

The base config matches the FedRBE app's expected `config.yml` format.
Per-center overrides set the correct `batch_col` (multi-batch vs single-batch centers).

In [21]:
base_config = {
    'flimmaBatchCorrection': {
        'data_filename': 'data.tsv',
        'design_filename': 'design.tsv',
        'expression_file_flag': True,
        'index_col': 0,
        'batch_col': None,
        'covariates': [],
        'separator': '\t',
        'design_separator': '\t',
        'normalizationMethod': None,
        'smpc': False,
        'min_samples': 0,
        'position': None,
    },
}

# Common overrides for all centers
common_changes = {
    'flimmaBatchCorrection': {
        'data_filename': 'intensities_log_UNION.tsv',
        'design_filename': 'design.tsv',
        'covariates': ['D6', 'F7', 'M8'],
        'index_col': 'rowname',
        'min_samples': 0,
    }
}

# Build per-center config changes
config_file_changes = []
for center in centers:
    changes = deepcopy(common_changes)
    if center in multi_batch_centers:
        changes['flimmaBatchCorrection']['batch_col'] = 'batch'
    else:
        changes['flimmaBatchCorrection']['batch_col'] = None
    config_file_changes.append(changes)

print('Per-center batch_col:')
for center, changes in zip(centers, config_file_changes):
    print(f"  {center}: {changes['flimmaBatchCorrection']['batch_col']}")

Per-center batch_col:
  APT: batch
  FDU: batch
  NVG: None
  BGI: None


## Helper: postprocessing

Extracts per-client results from ZIP files, concatenates into a single corrected matrix.

In [22]:
def postprocess_results(exp, result_files_zipped, result_filename):
    """Extract per-client results and concatenate into a single corrected TSV."""
    time.sleep(10)
    result_folder = os.path.dirname(result_filename)
    individual_results_dir = os.path.join(result_folder, 'individual_results')
    os.makedirs(individual_results_dir, exist_ok=True)

    client_names = [os.path.basename(p) for p in exp.clients]

    # Extract results from each client's zip
    # result_files_zipped already contains absolute paths from run_test()
    for zip_path, client_name in zip(result_files_zipped, client_names):
        # Wait for file to be available
        for _ in range(60):
            if os.path.exists(zip_path):
                break
            print(f'Waiting for {zip_path}...')
            time.sleep(5)
        else:
            raise RuntimeError(f'Zip file not found after timeout: {zip_path}')

        client_result_dir = os.path.join(individual_results_dir, client_name)
        os.makedirs(client_result_dir, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zip_names = zf.namelist()
            for fname in ['only_batch_corrected_data.csv', 'full_corrected_data.csv', 'report.txt']:
                if fname in zip_names:
                    zf.extract(fname, client_result_dir)
                elif fname == 'full_corrected_data.csv':
                    raise RuntimeError(
                        f"'{fname}' not found in zip for client '{client_name}'. "
                        f"Files in zip: {zip_names}"
                    )
                else:
                    print(f"WARNING: '{fname}' not in zip for '{client_name}', skipping.")

    # Update datainfo.json if present
    datainfo_path = os.path.join(result_folder, 'datainfo.json')
    if os.path.exists(datainfo_path):
        with open(datainfo_path, 'r') as f:
            datainfo = json.load(f)
        for cohort in datainfo.get('cohorts', []):
            cohort['folder'] = os.path.join('individual_results', cohort['name'])
        datainfo['datafile']['filename'] = 'full_corrected_data.csv'
        datainfo['datafile']['separator'] = '\t'
        with open(datainfo_path, 'w') as f:
            json.dump(datainfo, f, indent=2)
        print(f'Updated datainfo.json at {datainfo_path}')

    # Concatenate per-client corrected data
    final_df = None
    for client_name in client_names:
        result_file = os.path.join(individual_results_dir, client_name, 'only_batch_corrected_data.csv')
        if not os.path.exists(result_file):
            print(f'WARNING: Result file not found for {client_name}: {result_file}')
            continue
        client_df = pd.read_csv(result_file, sep='\t', index_col=0)
        final_df = client_df if final_df is None else pd.concat([final_df, client_df], axis=1)

    if final_df is None:
        raise RuntimeError('No data found in test results!')

    final_df.to_csv(result_filename, sep='\t')
    print(f'Saved concatenated result ({final_df.shape}) to {result_filename}')
    return final_df

## Run FedRBE (non-SMPC)

In [23]:
experiment = util.Experiment(
    name='Proteomics Multi Batch',
    fc_data_dir=multibatch_dir,
    clients=[os.path.join(before_dir, c) for c in centers],
    app_image_name=app_image_name,
    config_files=[deepcopy(base_config) for _ in centers],
    config_file_changes=deepcopy(config_file_changes),
)

# Add position argument (required for federated coordination)
for idx in range(len(experiment.clients)):
    experiment.config_file_changes[idx]['flimmaBatchCorrection']['position'] = idx

print(experiment)

Experiment(name='Proteomics Multi Batch', clients=['/home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/before/APT', '/home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/before/FDU', '/home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/before/NVG', '/home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/before/BGI'], app_image_name='featurecloud.ai/bcorrect:latest', fc_data_dir='/home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch', config_files=[{'flimmaBatchCorrection': {'data_filename': 'data.tsv', 'design_filename': 'design.tsv', 'expression_file_flag': True, 'index_col': 0, 'batch_col': None, 'covariates': [], 'separator': '\t', 'design_separator': '\t', 'normalizationMethod': None, 'smpc': False, 'min_samples': 0, 'position': None}}, {'flimmaBatchCorrection': {'data_filename': 'data.tsv', 'design_filename': 'design.tsv', 'expression_file_flag': T

In [24]:
result_files_zipped, _, _ = experiment.run_test()
print(f'Result files: {result_files_zipped}')

_______________EXPERIMENT_______________
Killing leftover containers...
Test started successfully!
Test finished successfully!
instances: [{'id': 0, 'name': 'fc_featurecloudaibcorrectlatest_659340757', 'containerID': '2973b6834e2a29f9', 'coordinator': True, 'frontendUrl': '/app-traffic/34f13d77ec/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 1, 'name': 'fc_featurecloudaibcorrectlatest_472227424', 'containerID': 'd367934573943765', 'coordinator': False, 'frontendUrl': '/app-traffic/cbad1ba271/', 'message': 'terminal', 'state': None, 'progress': None}, {'id': 2, 'name': 'fc_featurecloudaibcorrectlatest_370057443', 'containerID': '5452abe027ccc94c', 'coordinator': False, 'frontendUrl': '/app-traffic/f6c0a5bddf/', 'message': 'terminal', 'state': None, 'progress': None}, {'id': 3, 'name': 'fc_featurecloudaibcorrectlatest_439632798', 'containerID': '03cbd9c74b28410c', 'coordinator': False, 'frontendUrl': '/app-traffic/8c6743be01/', 'message': 'terminal', 'state': None, 'pro

In [25]:
result_path = os.path.join(after_dir, 'FedApp_corrected_data.tsv')
corrected_df = postprocess_results(experiment, result_files_zipped, result_path)
corrected_df.head()

Saved concatenated result ((3355, 72)) to /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/after/FedApp_corrected_data.tsv


,DDA_APT_D5_1,DDA_APT_D6_1,DDA_APT_F7_1,DDA_APT_M8_1,DDA_APT_D5_2,DDA_APT_D6_2,DDA_APT_F7_2,DDA_APT_M8_2,DDA_APT_D5_3,DDA_APT_D6_3,DDA_APT_F7_3,DDA_APT_M8_3,DIA_APT_D5_1,DIA_APT_D6_1,DIA_APT_F7_1,DIA_APT_M8_1,DIA_APT_D5_2,DIA_APT_D6_2,DIA_APT_F7_2,DIA_APT_M8_2,DIA_APT_D5_3,DIA_APT_D6_3,DIA_APT_F7_3,DIA_APT_M8_3,DDA_FDU_D5_1,DDA_FDU_D6_1,DDA_FDU_F7_1,DDA_FDU_M8_1,DDA_FDU_D5_2,DDA_FDU_D6_2,DDA_FDU_F7_2,DDA_FDU_M8_2,DDA_FDU_D5_3,DDA_FDU_D6_3,DDA_FDU_F7_3,DDA_FDU_M8_3,DIA_FDU_D5_1,DIA_FDU_D6_1,DIA_FDU_F7_1,DIA_FDU_M8_1,DIA_FDU_D5_2,DIA_FDU_D6_2,DIA_FDU_F7_2,DIA_FDU_M8_2,DIA_FDU_D5_3,DIA_FDU_D6_3,DIA_FDU_F7_3,DIA_FDU_M8_3,DDA_NVG_D5_1,DDA_NVG_D6_1,DDA_NVG_F7_1,DDA_NVG_M8_1,DDA_NVG_D5_2,DDA_NVG_D6_2,DDA_NVG_F7_2,DDA_NVG_M8_2,DDA_NVG_D5_3,DDA_NVG_D6_3,DDA_NVG_F7_3,DDA_NVG_M8_3,DIA_BGI_D5_1,DIA_BGI_D6_1,DIA_BGI_F7_1,DIA_BGI_M8_1,DIA_BGI_D5_2,DIA_BGI_D6_2,DIA_BGI_F7_2,DIA_BGI_M8_2,DIA_BGI_D5_3,DIA_BGI_D6_3,DIA_BGI_F7_3,DIA_BGI_M8_3
rowname,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Asparagine synthetase [glutamine-hydrolyzing],19.937818,19.939000,20.129547,19.906250,20.006961,19.944396,20.075645,19.601075,19.961292,20.009596,19.998406,19.785801,19.696819,19.746511,20.113314,19.587095,19.804456,20.044236,20.382665,19.778651,19.925294,20.125043,20.081408,20.010295,19.846808,20.025778,20.162513,19.715942,19.644960,19.925250,20.222824,19.838079,19.903367,20.054696,20.142942,19.812626,19.666154,19.961215,20.238288,19.768014,19.921102,20.073485,20.310243,19.699136,19.824775,19.997211,20.099074,19.737090,19.794633,19.882358,20.155653,19.796339,19.925757,19.637674,20.326441,19.846066,20.166346,19.915235,20.174036,19.675248,20.219926,19.975554,20.127315,19.626266,19.751307,19.809431,20.590511,19.689044,19.807974,19.913071,20.169014,19.616376
7-dehydrocholesterol reductase,18.635467,18.865225,18.974947,18.784178,18.702373,19.005675,18.842596,18.763351,18.752548,18.709812,18.713482,19.016522,18.760874,18.438155,18.995364,18.854942,18.815636,18.750842,19.071802,18.809180,18.769281,18.524784,19.020347,18.954967,18.675395,18.717815,18.969347,18.720124,18.621866,18.806867,18.790109,18.864241,18.683662,18.770379,18.968385,19.177984,18.656612,18.714612,18.954687,18.916176,18.842216,18.611502,18.903435,18.932676,18.744059,18.848854,18.762389,18.878956,18.656375,18.441716,18.861345,18.569403,18.558223,19.263159,19.024586,18.805303,18.799495,18.913018,18.945837,18.927717,18.900766,18.613803,18.808077,18.616548,18.529819,18.845808,19.489486,18.802736,18.448544,18.842693,18.838498,19.029397
Glycolipid transfer protein,18.060276,18.875243,18.003236,19.143228,17.891310,18.550433,17.461208,18.901295,17.855073,18.878342,18.852183,18.699282,17.412768,18.658115,18.233934,18.949981,18.162937,18.689136,18.421269,18.777577,18.599604,18.106598,18.683133,18.476057,18.353391,18.549515,18.251016,18.587317,18.033997,18.231108,18.424325,18.864900,18.285764,18.310910,18.513904,18.764962,18.252375,18.266954,18.400663,18.850652,18.407044,18.507976,18.539085,18.862256,18.182552,18.342100,18.002550,18.556903,18.310279,18.261932,17.739529,19.232416,NaN,18.191691,18.497891,18.497333,18.238371,19.207377,18.137794,18.689752,18.308608,18.466744,18.257102,18.463133,18.205732,18.380454,18.547035,18.930743,18.274574,18.080401,18.467498,18.789084
"39S ribosomal protein L44, mitochondrial",19.173313,18.603876,18.793967,18.757919,18.816275,18.581648,18.656495,18.568571,19.302783,18.565660,18.793323,18.440727,19.099435,18.847583,18.370553,18.495839,19.440758,18.787958,18.268622,18.778552,18.934435,18.702287,18.730018,18.598517,18.753765,18.635184,18.619676,18.957686,18.616514,18.630668,18.553939,18.728298,19.178644,19.002804,18.502370,18.875010,18.680797,19.085563,18.634882,18.693399,19.099009,18.743155,18.769933,18.478774,19.072205,18.702697,18.655435,18.438708,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19.155027,18.641476,18.602890,18.616846,18.885974,18.647521,19.114652,18.634249,18.920414,18.641460,18.547320,18.646729
Protein deglycase DJ-1,21.578416,

## Run FedRBE (SMPC)

Same experiment but with Secure Multi-Party Computation enabled.

In [26]:
experiment_smpc = util.Experiment(
    name='Proteomics Multi Batch (SMPC)',
    fc_data_dir=multibatch_dir,
    clients=[os.path.join(before_dir, c) for c in centers],
    app_image_name=app_image_name,
    config_files=[deepcopy(base_config) for _ in centers],
    config_file_changes=deepcopy(config_file_changes),
)

# Enable SMPC and set position
for idx in range(len(experiment_smpc.clients)):
    experiment_smpc.config_file_changes[idx]['flimmaBatchCorrection']['smpc'] = True
    experiment_smpc.config_file_changes[idx]['flimmaBatchCorrection']['position'] = idx

print(experiment_smpc)

Experiment(name='Proteomics Multi Batch (SMPC)', clients=['/home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/before/APT', '/home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/before/FDU', '/home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/before/NVG', '/home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/before/BGI'], app_image_name='featurecloud.ai/bcorrect:latest', fc_data_dir='/home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch', config_files=[{'flimmaBatchCorrection': {'data_filename': 'data.tsv', 'design_filename': 'design.tsv', 'expression_file_flag': True, 'index_col': 0, 'batch_col': None, 'covariates': [], 'separator': '\t', 'design_separator': '\t', 'normalizationMethod': None, 'smpc': False, 'min_samples': 0, 'position': None}}, {'flimmaBatchCorrection': {'data_filename': 'data.tsv', 'design_filename': 'design.tsv', 'expression_file_f

In [27]:
result_files_zipped_smpc, _, _ = experiment_smpc.run_test()
print(f'Result files: {result_files_zipped_smpc}')

_______________EXPERIMENT_______________
Killing leftover containers...
Test started successfully!
Test finished successfully!
instances: [{'id': 0, 'name': 'fc_featurecloudaibcorrectlatest_259351212', 'containerID': '720bdf8d01cc10ca', 'coordinator': False, 'frontendUrl': '/app-traffic/3277d77386/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 1, 'name': 'fc_featurecloudaibcorrectlatest_47468863', 'containerID': '2e0cd015d6dc5c92', 'coordinator': False, 'frontendUrl': '/app-traffic/e0f6b8f95b/', 'message': 'terminal', 'state': None, 'progress': None}, {'id': 2, 'name': 'fc_featurecloudaibcorrectlatest_916708054', 'containerID': 'c39c4fe761b5a3d7', 'coordinator': True, 'frontendUrl': '/app-traffic/60235b78e9/', 'message': 'terminal', 'state': None, 'progress': None}, {'id': 3, 'name': 'fc_featurecloudaibcorrectlatest_748823242', 'containerID': '339384be0e98f50e', 'coordinator': False, 'frontendUrl': '/app-traffic/f87906c7b7/', 'message': 'terminal', 'state': None, 'prog

In [28]:
result_path_smpc = os.path.join(after_dir, 'FedApp_corrected_data_smpc.tsv')
corrected_df_smpc = postprocess_results(experiment_smpc, result_files_zipped_smpc, result_path_smpc)
corrected_df_smpc.head()

Saved concatenated result ((3355, 72)) to /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/after/FedApp_corrected_data_smpc.tsv


,DDA_APT_D5_1,DDA_APT_D6_1,DDA_APT_F7_1,DDA_APT_M8_1,DDA_APT_D5_2,DDA_APT_D6_2,DDA_APT_F7_2,DDA_APT_M8_2,DDA_APT_D5_3,DDA_APT_D6_3,DDA_APT_F7_3,DDA_APT_M8_3,DIA_APT_D5_1,DIA_APT_D6_1,DIA_APT_F7_1,DIA_APT_M8_1,DIA_APT_D5_2,DIA_APT_D6_2,DIA_APT_F7_2,DIA_APT_M8_2,DIA_APT_D5_3,DIA_APT_D6_3,DIA_APT_F7_3,DIA_APT_M8_3,DDA_FDU_D5_1,DDA_FDU_D6_1,DDA_FDU_F7_1,DDA_FDU_M8_1,DDA_FDU_D5_2,DDA_FDU_D6_2,DDA_FDU_F7_2,DDA_FDU_M8_2,DDA_FDU_D5_3,DDA_FDU_D6_3,DDA_FDU_F7_3,DDA_FDU_M8_3,DIA_FDU_D5_1,DIA_FDU_D6_1,DIA_FDU_F7_1,DIA_FDU_M8_1,DIA_FDU_D5_2,DIA_FDU_D6_2,DIA_FDU_F7_2,DIA_FDU_M8_2,DIA_FDU_D5_3,DIA_FDU_D6_3,DIA_FDU_F7_3,DIA_FDU_M8_3,DDA_NVG_D5_1,DDA_NVG_D6_1,DDA_NVG_F7_1,DDA_NVG_M8_1,DDA_NVG_D5_2,DDA_NVG_D6_2,DDA_NVG_F7_2,DDA_NVG_M8_2,DDA_NVG_D5_3,DDA_NVG_D6_3,DDA_NVG_F7_3,DDA_NVG_M8_3,DIA_BGI_D5_1,DIA_BGI_D6_1,DIA_BGI_F7_1,DIA_BGI_M8_1,DIA_BGI_D5_2,DIA_BGI_D6_2,DIA_BGI_F7_2,DIA_BGI_M8_2,DIA_BGI_D5_3,DIA_BGI_D6_3,DIA_BGI_F7_3,DIA_BGI_M8_3
rowname,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Asparagine synthetase [glutamine-hydrolyzing],19.937818,19.939000,20.129547,19.906250,20.006961,19.944396,20.075645,19.601075,19.961292,20.009596,19.998406,19.785801,19.696819,19.746511,20.113314,19.587095,19.804456,20.044236,20.382665,19.778651,19.925294,20.125043,20.081408,20.010295,19.846808,20.025778,20.162513,19.715942,19.644960,19.925250,20.222824,19.838079,19.903367,20.054696,20.142942,19.812626,19.666154,19.961215,20.238288,19.768014,19.921102,20.073485,20.310243,19.699136,19.824775,19.997211,20.099074,19.737090,19.794633,19.882358,20.155653,19.796339,19.925757,19.637674,20.326441,19.846066,20.166346,19.915235,20.174036,19.675248,20.219926,19.975554,20.127315,19.626266,19.751307,19.809431,20.590511,19.689044,19.807974,19.913071,20.169014,19.616376
7-dehydrocholesterol reductase,18.635467,18.865225,18.974947,18.784178,18.702373,19.005675,18.842596,18.763351,18.752548,18.709812,18.713482,19.016522,18.760874,18.438155,18.995364,18.854942,18.815636,18.750842,19.071802,18.809180,18.769281,18.524784,19.020347,18.954967,18.675395,18.717815,18.969347,18.720124,18.621866,18.806867,18.790109,18.864241,18.683662,18.770379,18.968385,19.177984,18.656612,18.714612,18.954687,18.916176,18.842216,18.611502,18.903435,18.932676,18.744059,18.848854,18.762389,18.878956,18.656375,18.441716,18.861345,18.569403,18.558223,19.263159,19.024586,18.805303,18.799495,18.913018,18.945837,18.927717,18.900766,18.613803,18.808077,18.616548,18.529819,18.845808,19.489486,18.802736,18.448544,18.842693,18.838498,19.029397
Glycolipid transfer protein,18.060276,18.875243,18.003236,19.143228,17.891310,18.550433,17.461208,18.901295,17.855073,18.878342,18.852183,18.699282,17.412768,18.658115,18.233934,18.949981,18.162937,18.689136,18.421269,18.777577,18.599604,18.106598,18.683133,18.476057,18.353391,18.549515,18.251016,18.587317,18.033997,18.231108,18.424325,18.864900,18.285764,18.310910,18.513904,18.764962,18.252375,18.266954,18.400663,18.850652,18.407044,18.507976,18.539085,18.862256,18.182552,18.342100,18.002550,18.556903,18.310279,18.261932,17.739529,19.232416,NaN,18.191691,18.497891,18.497333,18.238371,19.207377,18.137794,18.689752,18.308608,18.466744,18.257102,18.463133,18.205732,18.380454,18.547035,18.930743,18.274574,18.080401,18.467498,18.789084
"39S ribosomal protein L44, mitochondrial",19.173313,18.603876,18.793967,18.757919,18.816275,18.581648,18.656495,18.568571,19.302783,18.565660,18.793323,18.440727,19.099435,18.847583,18.370553,18.495839,19.440758,18.787958,18.268622,18.778552,18.934435,18.702287,18.730018,18.598517,18.753765,18.635184,18.619676,18.957686,18.616514,18.630668,18.553939,18.728298,19.178644,19.002804,18.502370,18.875010,18.680797,19.085563,18.634882,18.693399,19.099009,18.743155,18.769933,18.478774,19.072205,18.702697,18.655435,18.438708,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19.155027,18.641476,18.602890,18.616846,18.885974,18.647521,19.114652,18.634249,18.920414,18.641460,18.547320,18.646729
Protein deglycase DJ-1,21.578416,

## Verify outputs

In [29]:
for fname in ['FedApp_corrected_data.tsv', 'FedApp_corrected_data_smpc.tsv']:
    fpath = os.path.join(after_dir, fname)
    if os.path.exists(fpath):
        df = pd.read_csv(fpath, sep='\t', index_col=0)
        print(f'{fname}: {df.shape[0]} features x {df.shape[1]} samples')
    else:
        print(f'{fname}: NOT FOUND')

# Check individual results
ind_dir = os.path.join(after_dir, 'individual_results')
if os.path.exists(ind_dir):
    for center in centers:
        for fname in ['full_corrected_data.csv', 'only_batch_corrected_data.csv', 'report.txt']:
            fpath = os.path.join(ind_dir, center, fname)
            status = 'OK' if os.path.exists(fpath) else 'MISSING'
            print(f'  {center}/{fname}: {status}')

FedApp_corrected_data.tsv: 3355 features x 72 samples
FedApp_corrected_data_smpc.tsv: 3355 features x 72 samples
  APT/full_corrected_data.csv: OK
  APT/only_batch_corrected_data.csv: OK
  APT/report.txt: OK
  FDU/full_corrected_data.csv: OK
  FDU/only_batch_corrected_data.csv: OK
  FDU/report.txt: OK
  NVG/full_corrected_data.csv: OK
  NVG/only_batch_corrected_data.csv: OK
  NVG/report.txt: OK
  BGI/full_corrected_data.csv: OK
  BGI/only_batch_corrected_data.csv: OK
  BGI/report.txt: OK


## Cleanup: remove tests and logs folders, stop controller, prune Docker volumes

In [35]:
import shutil
import subprocess
from FeatureCloud.api.imp.controller import commands as fc_controller

# Stop the FeatureCloud controller first (so it releases file locks)
try:
    fc_controller.stop(name=fc_controller.DEFAULT_CONTROLLER_NAME)
    print('FeatureCloud controller stopped.')
except Exception as e:
    print(f'Controller stop: {e}')

# Remove tests and logs folders using Docker (files are owned by root from container runs)
tests_dir = os.path.join(multibatch_dir, 'tests')
logs_dir = os.path.join(multibatch_dir, 'logs')
workflows_dir = os.path.join(multibatch_dir, 'workflows')

for folder in [tests_dir, logs_dir, workflows_dir]:
    if os.path.exists(folder):
        # Use a Docker container to delete contents as root (rm the contents, not the mount point)
        subprocess.run(
            ['docker', 'run', '--rm', '-v', f'{folder}:/cleanup', 'alpine', 'sh', '-c', 'rm -rf /cleanup/*'],
            check=True,
        )
        # Now the folder is empty and owned by current user, safe to remove
        shutil.rmtree(folder, ignore_errors=True)
        print(f'Removed: {folder}')
    else:
        print(f'Not found (already clean): {folder}')

# Remove dangling Docker volumes created by tests
import docker
client = docker.from_env()
pruned = client.volumes.prune()
freed = pruned.get('SpaceReclaimed', 0)
print(f'Pruned Docker volumes. Space reclaimed: {freed / 1e9:.2f} GB')

# plus remove site_info.json if exists
site_info_path = os.path.join(multibatch_dir, 'site_info.json')
if os.path.exists(site_info_path):
    os.remove(site_info_path)
    print(f'Removed: {site_info_path}')
else:
    print(f'Not found (already clean): {site_info_path}') 

FeatureCloud controller stopped.
Not found (already clean): /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/tests
Not found (already clean): /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/logs
Not found (already clean): /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/workflows
Pruned Docker volumes. Space reclaimed: 0.00 GB
Removed: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/proteomics_multibatch/site_info.json
